In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("Mall_Customers.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Statistik deskriptif dataset
df.describe()

In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']]

In [ ]:
X.head()

## Data Preprocessing

Sebelum melakukan clustering, data Annual Income dan Spending Score dilakukan standardisasi menggunakan StandardScaler.

Standardisasi dilakukan agar kedua fitur memiliki skala yang sebanding sehingga tidak ada fitur yang terlalu dominan dalam proses clustering.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
print(X_scaled[:5])

## Menentukan Jumlah Cluster

Untuk menentukan jumlah cluster yang optimal, digunakan metode Elbow.

Metode ini dilakukan dengan membandingkan nilai inertia dari beberapa jumlah cluster. Jumlah cluster dipilih berdasarkan titik ketika penurunan inertia mulai melandai.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia, marker='o')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.xticks(range(1, 11))
plt.grid()
plt.show()

## Pemodelan K-Means

Berdasarkan hasil Elbow Method, jumlah cluster yang digunakan adalah 5.

Selanjutnya, algoritma K-Means digunakan untuk mengelompokkan pelanggan berdasarkan Annual Income dan Spending Score.

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)

cluster = kmeans.fit_predict(X_scaled)

df['Cluster'] = cluster

df.head()

## Profiling Cluster

Setelah setiap pelanggan mendapatkan label cluster, dilakukan analisis rata-rata Age, Annual Income, dan Spending Score pada setiap cluster.

Profiling ini digunakan untuk memahami karakteristik masing-masing kelompok pelanggan.

In [ ]:
cluster_profile = df.groupby('Cluster')[
    ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
].mean().round(2)

cluster_profile

## Interpretasi Hasil Clustering

Berdasarkan hasil profiling, diperoleh lima kelompok pelanggan dengan karakteristik yang berbeda:

### Cluster 0 — Moderate Customer
Memiliki rata-rata pendapatan sebesar 55.30 k$ dan Spending Score sebesar 49.52. Kelompok ini menunjukkan tingkat pendapatan dan belanja yang relatif sedang.

### Cluster 1 — High Income - High Spending
Memiliki rata-rata pendapatan sebesar 86.54 k$ dan Spending Score sebesar 82.13. Kelompok ini memiliki pendapatan tinggi dan tingkat belanja yang tinggi.

### Cluster 2 — Young High-Spending Customer
Memiliki rata-rata pendapatan sebesar 25.73 k$ dan Spending Score sebesar 79.36. Kelompok ini memiliki pendapatan relatif rendah tetapi tingkat belanja yang tinggi.

### Cluster 3 — High Income - Low Spending
Memiliki rata-rata pendapatan sebesar 88.20 k$ dan Spending Score sebesar 17.11. Kelompok ini memiliki pendapatan tinggi tetapi tingkat belanja yang rendah.

### Cluster 4 — Low Income - Low Spending
Memiliki rata-rata pendapatan sebesar 26.30 k$ dan Spending Score sebesar 20.91. Kelompok ini memiliki pendapatan dan tingkat belanja yang relatif rendah.

## Visualisasi Hasil Clustering

Visualisasi digunakan untuk melihat persebaran pelanggan berdasarkan Annual Income dan Spending Score serta perbedaan antar-cluster.

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    c=df['Cluster'],
    s=50
)

plt.xlabel('Annual Income (Scaled)')
plt.ylabel('Spending Score (Scaled)')
plt.title('Customer Segmentation menggunakan K-Means')

plt.show()

## Evaluasi Model

Kualitas hasil clustering dievaluasi menggunakan Silhouette Score.

Silhouette Score digunakan untuk melihat seberapa baik setiap data berada dalam cluster-nya dibandingkan dengan cluster lainnya. Nilai yang semakin mendekati 1 menunjukkan pemisahan cluster yang semakin baik.

In [ ]:
from sklearn.metrics import silhouette_score

silhouette = silhouette_score(X_scaled, df['Cluster'])

print("Silhouette Score:", round(silhouette, 3))

### Hasil Evaluasi

Model menghasilkan Silhouette Score sebesar **0.555**. Nilai tersebut menunjukkan bahwa cluster yang terbentuk memiliki pemisahan yang cukup baik berdasarkan fitur Annual Income dan Spending Score.

In [ ]:
cluster_profile = df.groupby('Cluster')[
    ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
].mean().round(2)

cluster_profile

## Export Model dan Scaler

Model K-Means dan StandardScaler disimpan dalam format `.pkl` menggunakan library Joblib.

File tersebut digunakan kembali pada aplikasi Flask agar model yang sudah dilatih dapat digunakan untuk melakukan segmentasi pada data pelanggan baru.

In [ ]:
import joblib

joblib.dump(kmeans, 'kmeans_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Model berhasil disimpan!")

In [ ]:
import joblib
import pandas as pd

# Load model dan scaler
model = joblib.load("kmeans_model.pkl")
scaler_test = joblib.load("scaler.pkl")

# Data customer baru
new_customer = pd.DataFrame(
    [[86, 82]],
    columns=['Annual Income (k$)', 'Spending Score (1-100)']
)

# Scaling
new_customer_scaled = scaler_test.transform(new_customer)

# Prediksi cluster
predicted_cluster = model.predict(new_customer_scaled)

print("Predicted Cluster:", predicted_cluster[0])